In [56]:
# pip install faker
# pip install faker-food

## Import Libraries

In [106]:
import psycopg2
import os
from dotenv import load_dotenv
from faker import Faker
from faker_food import FoodProvider
import random
from psycopg2.extras import execute_values


## Cloud Database Connection

In [107]:
# connect to the PostgreSQL database
load_dotenv()
conn = psycopg2.connect (
    dbname = os.getenv("DB_NAME"),
    user = os.getenv("DB_USER"),
    password = os.getenv("DB_PASSWORD"),
    host = os.getenv("DB_HOST"),
    port = os.getenv("DB_PORT")
)

cur = conn.cursor()

In [95]:
print(cur)

<cursor object at 0x000002BA57B0AF80; closed: 0>


## Data Population with Faker 

In [108]:
conn.rollback()

#### Data popluation for stores Table

In [40]:
# Define the number of records
number_records = 5

# instantiate Faker object
fake = Faker()

records = []

# Generate and insert data into the store table
for _ in range(number_records):
    address = fake.address()
    city = fake.city()
    phone_number = fake.phone_number()
    opened_at = fake.date_time()
    records.append((address, city, phone_number, opened_at))

# Construct the sql with placeholder
statement = "INSERT INTO public.stores(address, city, phone_number, opened_at) VALUES (%s, %s, %s, %s)"

#Execute the SQL statement with the Values
cur.executemany(statement, records)

# Commit the transaction
conn.commit()
print("Stores Data Inserted Successfully!")

Stores Data Inserted Successfully!


#### Data Population For Customers Table

In [ ]:
number_records = 1000

faker = Faker()

emails = set()
records = []

# Generate and insert Data into customer Table
for _ in range(number_records):
    first_name = faker.first_name()
    lastname = faker.last_name()
    email = faker.email()
    if email in emails: # skip duplicates
        continue
    emails.add(email)
    phone_number = faker.phone_number()
    created_at = faker.date_time()

    MAX_LENGTHS = {
        "first_name" : 100,
        "lastname" : 100,
        "email" : 100,
        "phone_number" : 50,
    }
    # records.append((
    #     first_name[:MAX_LENGTHS["first_name"]],
    #     lastname[:MAX_LENGTHS["lastname"]],
    #     email[:MAX_LENGTHS["email"]],
    #     phone_number[:MAX_LENGTHS["phone_number"]],
    #     created_at
    #     ))

    records.append((first_name, lastname, email, phone_number, created_at))



# Construct SQL Statemnt with Placeholder
statement = "INSERT INTO public.customers(first_name, lastname, email, phone_number, created_at) VALUES (%s, %s, %s, %s, %s)"

# Insert in batches
# batch_size = 200
# for i in range(0, len(records), batch_size):
#     cur.executemany(statement, records[i:i+batch_size])

# Execute the SQL statement with Values
cur.executemany(statement, records)

# Commit the transaction
conn.commit()
print("Customers Data has been inserted successfully!")

Customers Data has been inserted successfully!


#### Data Population For Ingredients Table

In [75]:
number_records = 50

faker = Faker()
faker.add_provider(FoodProvider)

records = []
units = ["g", "kg", "ml", "l", "tsp", "tbsp"]

# Generate and insert Data into customer Table
for _ in range(number_records):
    name = faker.ingredient()
    stock_quantity = fake.random_int(min=1, max=200)
    unit = faker.random_element(units)
    records.append((name, stock_quantity, unit))

    statement = "INSERT INTO public.ingredients(name, stock_quantity, unit) VALUES (%s, %s, %s)"


# Execute the SQL statement with Values

cur.executemany(statement, records)

# Commit transaction
conn.commit()

print("Ingredients Data has been inserted successfully!")


Ingredients Data has been inserted successfully!


#### Data Population For MENU_ITEMS Table

In [89]:
faker = Faker()
faker.add_provider(FoodProvider)
number_records = 30
records = []
categories = ["Appetizer", "Meal Deal", "Snack", "Juice", "Wine", "Belle Full", "Main Course", "Dessert", "Side Dish", "Beverage", "Salad", "Soup"]
sizes = ["small", "medium", "Large"]
for _ in range(number_records):
    name = faker.dish()
    category = faker.random_element(categories)
    item_price = round(faker.random.uniform(10, 550), 2)
    size = faker.random_element(sizes)

    records.append((name, category, item_price, size))

    statement = "INSERT INTO public.menu_items(name, category, item_price, size) VALUES (%s, %s, %s, %s)"

# Execute the SQL statement with Values
cur.executemany(statement, records)

conn.commit()

print("Transaction for Munu_Items completed Successfully!")


Transaction for Munu_Items completed Successfully!


#### Data Population For ORDERS Table

In [100]:
faker = Faker()
number_records = 5000
records = []

# Get valid foreign keys
cur.execute('SELECT customer_id FROM customers')
customer_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT store_id FROM stores")
store_ids = [row[0] for row in cur.fetchall()]

for _ in range(number_records):
    customer_id = faker.random_element(customer_ids)
    store_id = faker.random_element(store_ids)
    order_timestamp = faker.date_time()
    total_amount = round(faker.random.uniform(10, 200), 2)

    records.append((customer_id, store_id, order_timestamp, total_amount))

statement = "INSERT INTO orders (customer_id, store_id, order_timestamp, total_amount) VALUES (%s, %s, %s, %s)"


cur.executemany(statement, records)
conn.commit()

print("Transaction for orders completed Successfully!")

Transaction for orders completed Successfully!


#### Data Population for order_items Table

In [111]:
faker = Faker()
number_records = 15000
records = []

cur.execute("SELECT order_id FROM orders")
order_ids = [row[0] for row in cur.fetchall()]

cur.execute("SELECT item_id FROM menu_items")
item_ids = [row[0] for row in cur.fetchall()]

#Safety Check
if not order_ids or not item_ids:
    raise Exception("Orders or Menu Items is empty")

# Generate Exact number_records row
 

for _ in range(number_records):
    order_id = random.choice(order_ids)
    item_id = random.choice(item_ids)
    quantity = random.randint(1, 3)
    unit_price = round(faker.random.uniform(5, 25), 2)

    records.append((order_id, item_id, quantity, unit_price))

statement = "INSERT INTO order_items(order_id, item_id, quantity, unit_price) VALUES %s"

execute_values(cur, statement, records)
conn.commit()
print(f"{number_records} order_items inserted Successfully!")

15000 order_items inserted Successfully!
